In [ ]:
from datasets import load_dataset
ds = load_dataset("SimulaMet-HOST/Kvasir-VQA")["raw"]

In [2]:
ds

Dataset({
    features: ['image', 'source', 'question', 'answer', 'img_id'],
    num_rows: 58849
})

In [3]:
import os
from datasets import Dataset, Features, Image, Value, load_dataset, DatasetDict
from PIL import Image as PILImage
import random
import pandas as pd
from collections import defaultdict

In [4]:
from datasets import concatenate_datasets

# Remove invalid question entries
valid_ds = ds.filter(lambda ex: ex["question"] and ex["question"] != "none")

# Identify abnormal samples (source != 'normal')
abnormal_ids = set(
    ex["img_id"] for ex in valid_ds if ex["source"].lower() != "normal"
)

# add "Does this image contain any finding?" = "yes" for abnormal cases
seen_ids = set()
added_examples = []
for ex in valid_ds:
    if ex["img_id"] in abnormal_ids and ex["img_id"] not in seen_ids:
        added_examples.append({
            "image": ex["image"],
            "source": ex["source"],
            "question": "Does this image contain any finding?",
            "answer": "yes",
            "img_id": ex["img_id"]
        })
        seen_ids.add(ex["img_id"])

# Combine cleaned data with added questions
modified_ds = Dataset.from_list(added_examples)
cleaned_ds = concatenate_datasets([valid_ds, modified_ds])

In [5]:
# summary
print("Original size:", len(ds))
print("Cleaned size: ", len(valid_ds))
print("final modifierd data size: ", len(cleaned_ds))
print("new added QAs: ", len(added_examples))

Original size: 58849
Cleaned size:  58798
final modifierd data size:  62747
new added QAs:  3949


In [6]:
cleaned_ds

Dataset({
    features: ['image', 'source', 'question', 'answer', 'img_id'],
    num_rows: 62747
})

In [7]:
from collections import defaultdict

question_answers = defaultdict(set)

# Loop through all training examples
for example in cleaned_ds:
    q = example['question']
    a = example['answer']
    
    # If answer is list (sometimes it is), add each
    if isinstance(a, list):
        question_answers[q].update(a)
    else:
        question_answers[q].add(a)

# Print all unique questions with their answer sets
for i, (q, answers) in enumerate(question_answers.items(), 1):
    print(f"Q{i}: {q}")
    print(f"Possible Answers ({len(answers)}): {sorted(list(answers))}\n")

Q1: Are there any abnormalities in the image? Check all that are present.
Possible Answers (10): ['barretts', 'barretts; oesophagitis', 'hemorrhoids', 'oesophagitis', 'oesophagitis; polyp', 'oesophagitis; short-segment barretts', 'polyp', 'polyp; short-segment barretts', 'polyp; ulcerative colitis', 'ulcerative colitis']

Q2: Are there any anatomical landmarks in the image? Check all that are present.
Possible Answers (5): ['cecum', 'ileum', 'none', 'pylorus', 'z-line']

Q3: Are there any instruments in the image? Check all that are present.
Possible Answers (11): ['biopsy forceps', 'biopsy forceps; metal clip', 'biopsy forceps; tube', 'injection needle', 'metal clip', 'metal clip; polyp snare', 'metal clip; tube', 'none', 'polyp snare', 'polyp snare; tube', 'tube']

Q4: Have all polyps been removed?
Possible Answers (3): ['no', 'not relevant', 'yes']

Q5: Is this finding easy to detect?
Possible Answers (3): ['no', 'not relevant', 'yes']

Q6: Is there a green/black box artefact?
Possi

In [7]:
import os
from datasets import Dataset, Features, Image, Value, load_dataset, DatasetDict
from PIL import Image as PILImage

In [9]:
for example in ds.select(range(5)): 
    print(f"Image:  {example['image']!r}")
    print(f"Source:  {example['source']!r}")
    print(f"Question: {example['question']!r}")
    print(f"Answer: {example['answer']!r}")
    print(f"Image Id: {example['img_id']!r}")
    print('-'*40)

Image:  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=720x576 at 0x75FF182524B0>
Source:  'Ulcerative Colitis'
Question: 'Are there any abnormalities in the image? Check all that are present.'
Answer: 'ulcerative colitis'
Image Id: 'cla820gl0s3nv071u4fgd7xgq'
----------------------------------------
Image:  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=720x576 at 0x75FF18252510>
Source:  'Ulcerative Colitis'
Question: 'Are there any anatomical landmarks in the image? Check all that are present.'
Answer: 'none'
Image Id: 'cla820gl0s3nv071u4fgd7xgq'
----------------------------------------
Image:  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=720x576 at 0x75FF182524E0>
Source:  'Ulcerative Colitis'
Question: 'Are there any instruments in the image? Check all that are present.'
Answer: 'none'
Image Id: 'cla820gl0s3nv071u4fgd7xgq'
----------------------------------------
Image:  <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=720x576 at 0x75FF182519A0>
Sou

In [8]:
import random
from datasets import load_dataset, DatasetDict

# all unique img_ids
all_ids = sorted(set(cleaned_ds["img_id"]))

# Shuffle & split those IDs into 80/10/10
random.seed(42)
random.shuffle(all_ids)

n = len(all_ids)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
# n_test will be whatever is left
train_ids = set(all_ids[:n_train])
val_ids   = set(all_ids[n_train : n_train + n_val])
test_ids  = set(all_ids[n_train + n_val :])



In [9]:
def filter_by_id(example, id_set):
    return example["img_id"] in id_set


In [10]:
train_ds = cleaned_ds.filter(lambda ex: filter_by_id(ex, train_ids), 
                     batched=False)
val_ds   = cleaned_ds.filter(lambda ex: filter_by_id(ex, val_ids), 
                     batched=False)
test_ds  = cleaned_ds.filter(lambda ex: filter_by_id(ex, test_ids), 
                     batched=False)

In [11]:
dataset = DatasetDict({
    "train":      train_ds,
    "validation": val_ds,
    "test":       test_ds,
})

print({k: len(v) for k, v in dataset.items()})

{'train': 50105, 'validation': 6287, 'test': 6355}


In [12]:
# keeping copy for tracking ( image,question,answer,source,img_id)  #dataset_full['test'][i]['img_id'] (or ['source'])
dataset_full = dataset

data = dataset_full.remove_columns(["source", "img_id"])
data

DatasetDict({
    train: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 50105
    })
    validation: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6287
    })
    test: Dataset({
        features: ['image', 'question', 'answer'],
        num_rows: 6355
    })
})

In [15]:
data['train'].features

{'image': Image(mode=None, decode=True, id=None),
 'question': Value(dtype='string', id=None),
 'answer': Value(dtype='string', id=None)}

In [16]:
data['train'][0]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=720x576>,
 'question': 'Are there any abnormalities in the image? Check all that are present.',
 'answer': 'ulcerative colitis'}

In [17]:
dataset_full['test']

Dataset({
    features: ['image', 'source', 'question', 'answer', 'img_id'],
    num_rows: 6355
})

In [14]:
from transformers import AutoModelForCausalLM, AutoProcessor
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True).to(device) #if unable to install flash_attn, then execute the below code to load the model

# with patch("transformers.dynamic_module_utils.get_imports", fixed_get_imports): #workaround for unnecessary flash_attn requirement
#             model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-large", attn_implementation="sdpa",trust_remote_code=True).to(device)


processor = AutoProcessor.from_pretrained("microsoft/Florence-2-large", trust_remote_code=True)

/home/ebmi/anaconda3/envs/kvasir-vlm/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [15]:
torch.cuda.empty_cache()

In [16]:
def print_trainable_parameters(model):
    trainable = 0
    total = 0
    for name, param in model.named_parameters():
        total += param.numel()
        if param.requires_grad:
            trainable += param.numel()
    print(f"Trainable parameters: {trainable:,}")
    print(f"Total parameters: {total:,}")
    print(f"Percentage of trainable parameters: {100 * trainable / total:.2f}%")

print_trainable_parameters(model)

Trainable parameters: 828,985,344
Total parameters: 828,985,344
Percentage of trainable parameters: 100.00%


### Constructing the dataset for finetuning

In [17]:
from torch.utils.data import Dataset

class DocVQADataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        question = "<DocVQA>" + example['question']
        first_answer = example['answer'] # not example['answer'][0]
        image = example['image'].resize([384, 384]) #according t the original paper of florence-2 
        if image.mode != "RGB":
            image = image.convert("RGB")
        return question, first_answer, image

In [18]:
import os
from torch.utils.data import DataLoader
from tqdm import tqdm

def collate_fn(batch):
    questions, answers, images = zip(*batch)
    inputs = processor(
        text=list(questions),
        images=list(images),
        return_tensors="pt",
        padding=True,
        do_resize=False #otherwise processor will further resize it to 768x768
    ).to(device)
    return inputs, answers

# Create datasets
train_dataset = DocVQADataset(dataset['train'])
val_dataset = DocVQADataset(dataset['validation'])

# Create DataLoader
batch_size = 8 # 1 if T4(or similar gpu)
num_workers = 0

train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fn, num_workers=num_workers, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=collate_fn, num_workers=num_workers)

In [19]:
from transformers import (AutoProcessor, get_scheduler)
from torch.optim import AdamW

def train_model(train_loader, val_loader, model, processor, epochs=10, lr=1e-6):
    optimizer = AdamW(model.parameters(), lr=lr)
    num_training_steps = epochs * len(train_loader)
    lr_scheduler = get_scheduler(
        name="linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        i = -1
        for batch in tqdm(train_loader, desc=f"Training Epoch {epoch + 1}/{epochs}"):
            i += 1
            inputs, answers = batch

            input_ids = inputs["input_ids"]
            pixel_values = inputs["pixel_values"]
            labels = processor.tokenizer(text=answers, return_tensors="pt", padding=True, return_token_type_ids=False).input_ids.to(device)

            outputs = model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_loader)
        print(f"Average Training Loss: {avg_train_loss}")

        # Validation phase
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Validation Epoch {epoch + 1}/{epochs}"):
                inputs, answers = batch

                input_ids = inputs["input_ids"]
                pixel_values = inputs["pixel_values"]
                labels = processor.tokenizer(text=answers, return_tensors="pt", padding=True, return_token_type_ids=False).input_ids.to(device)

                outputs = model(input_ids=input_ids, pixel_values=pixel_values, labels=labels)
                loss = outputs.loss

                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_loader)
        print(f"Average Validation Loss: {avg_val_loss}")

        # Save model checkpoint
        output_dir = f"florence2_large_checkpoints/epoch_{epoch+1}"
        os.makedirs(output_dir, exist_ok=True)
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)


In [27]:
train_model(train_loader, val_loader, model, processor, epochs=10)

Training Epoch 1/10: 100%|██████████| 6264/6264 [1:26:34<00:00,  1.21it/s]


Average Training Loss: 0.3600629021618132


Validation Epoch 1/10: 100%|██████████| 786/786 [03:59<00:00,  3.28it/s]


Average Validation Loss: 0.28676693071315


Training Epoch 2/10: 100%|██████████| 6264/6264 [1:25:03<00:00,  1.23it/s]


...

Average Validation Loss: 0.25543341354369814


Training Epoch 10/10: 100%|██████████| 6264/6264 [1:25:00<00:00,  1.23it/s]


Average Training Loss: 0.029710827029395256


Validation Epoch 10/10: 100%|██████████| 786/786 [03:57<00:00,  3.31it/s]


Average Validation Loss: 0.2572823433020643


## Test

In [20]:
test_dataset = DocVQADataset(dataset['test'])
test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=collate_fn, num_workers=num_workers)

In [22]:
def evaluate_model(test_loader, model, processor):
    model.eval()
    correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            inputs, true_answers = batch

            input_ids = inputs["input_ids"]
            pixel_values = inputs["pixel_values"]

            # print(input_ids.shape)
            # print(pixel_values.shape)

            # Generate predictions from the model
            outputs = model.generate(input_ids=input_ids, pixel_values=pixel_values, max_new_tokens=50)

            # Decode the generated tokens to text
            predicted_answers = processor.tokenizer.batch_decode(outputs, skip_special_tokens=True)

            # Compare each predicted answer to the ground-truth answer
            for true_answer, predicted_answer in zip(true_answers, predicted_answers):
                if predicted_answer.strip().lower() == true_answer.strip().lower():
                    correct_predictions += 1
                total_predictions += 1

    accuracy = correct_predictions / total_predictions * 100
    return accuracy

In [ ]:
# Run the evaluation
test_accuracy = evaluate_model(test_loader, model, processor)
print(f"Test Accuracy: {test_accuracy:.2f}%")

In [32]:
## import os
from safetensors.torch import load_file
import torch

# Define the checkpoint directory where the model checkpoints are saved
checkpoint_dir = "florence2_large_checkpoints"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Loop through all the epoch directories and evaluate the model at each checkpoint
for epoch_dir in sorted(os.listdir(checkpoint_dir)):
    epoch_path = os.path.join(checkpoint_dir, epoch_dir, "model.safetensors")

    if os.path.exists(epoch_path):
        print(f"Loading weights from: {epoch_path}")

        # Load the state dict from safetensors
        state_dict = load_file(epoch_path)

        # Load the state dict into the model
        model.load_state_dict(state_dict, strict=False)
        model = model.to(device)

        # Evaluate the model for the current epoch
        test_accuracy = evaluate_model(test_loader, model, processor)

        # Print test accuracy for the current epoch
        print(f"(Epoch {epoch_dir}) Test Accuracy: {test_accuracy:.2f}%")


Loading weights from: florence2_large_checkpoints/epoch_1/model.safetensors


Evaluating: 100%|██████████| 795/795 [05:11<00:00,  2.55it/s]


(Epoch epoch_1) Test Accuracy: 82.05%
Loading weights from: florence2_large_checkpoints/epoch_10/model.safetensors


Evaluating: 100%|██████████| 795/795 [05:03<00:00,  2.62it/s]


(Epoch epoch_10) Test Accuracy: 84.82%
Loading weights from: florence2_large_checkpoints/epoch_2/model.safetensors


...

(Epoch epoch_7) Test Accuracy: 84.42%
Loading weights from: florence2_large_checkpoints/epoch_8/model.safetensors


Evaluating: 100%|██████████| 795/795 [05:04<00:00,  2.61it/s]


(Epoch epoch_8) Test Accuracy: 84.85%
Loading weights from: florence2_large_checkpoints/epoch_9/model.safetensors


Evaluating: 100%|██████████| 795/795 [04:58<00:00,  2.66it/s]

(Epoch epoch_9) Test Accuracy: 85.16%


In [24]:
from safetensors.torch import load_file

state_dict = load_file('/home/ebmi/Desktop/Research/Codes/kvasir-vqa/florence2_large_checkpoints/epoch_9/model.safetensors')

model.load_state_dict(state_dict, strict=False)
model = model.to(device)

test_accuracy = evaluate_model(test_loader, model, processor)
print(f"Test Accuracy: {test_accuracy:.2f}%")

Evaluating: 100%|██████████| 795/795 [05:01<00:00,  2.64it/s]

Test Accuracy: 85.16%


In [26]:
import torch
import evaluate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import Levenshtein

def evaluate_model(test_loader, model, processor):
    model.eval()
    preds, refs = [], []
    correct, total = 0, 0

    # run inference and collect everything
    with torch.no_grad():
        for inputs, true_answers in tqdm(test_loader, desc="Evaluating"):
            input_ids = inputs["input_ids"]
            pixel_values = inputs["pixel_values"]
            # generate
            outputs = model.generate(
                input_ids=input_ids,
                pixel_values=pixel_values,
                max_new_tokens=50,
                num_beams=5,
                early_stopping=True
            )
            # decode
            batch_preds = processor.tokenizer.batch_decode(outputs, skip_special_tokens=True)
            # accumulate
            for true, pred in zip(true_answers, batch_preds):
                refs.append(true.strip())
                p = pred.strip()
                preds.append(p)
                if p.lower() == true.strip().lower():
                    correct += 1
                total += 1

    # Accuracy
    accuracy = correct / total * 100

    # load HF metrics
    bleu  = evaluate.load("bleu")
    rouge = evaluate.load("rouge")
    meteor= evaluate.load("meteor")

    bleu_res   = bleu.compute(predictions=preds, references=[[r] for r in refs])
    rouge_res  = rouge.compute(predictions=preds, references=refs)
    meteor_res = meteor.compute(predictions=preds, references=refs)

    # Jaccard similarity (token‐set Jaccard)
    j_scores = []
    for r, p in zip(refs, preds):
        set_r, set_p = set(r.split()), set(p.split())
        if set_r or set_p:
            j_scores.append(len(set_r & set_p) / len(set_r | set_p))
    jaccard = sum(j_scores) / len(j_scores) * 100

    # Cosine similarity via TF‐IDF
    vectorizer = TfidfVectorizer().fit(refs + preds)
    ref_vecs  = vectorizer.transform(refs)
    pred_vecs = vectorizer.transform(preds)
    cos_sims  = cosine_similarity(ref_vecs, pred_vecs).diagonal()
    cosine    = cos_sims.mean() * 100

    ###added

    # Levenshtein Similarity
    def normalized_levenshtein(s1, s2):
        if not s1 and not s2:
            return 0
        return Levenshtein.distance(s1, s2) / max(len(s1), len(s2))

    def similarity_score(ref, pred, tau=0.5):
        nl = normalized_levenshtein(ref, pred)
        return 1 - nl if nl < tau else 0

    def average_levenshtein_similarity(ground_truth, predicted):
        total_score = 0
        for ref, pred in zip(ground_truth, predicted):
            if not pred:
                continue
            score = similarity_score(ref, pred)
            total_score += score
        return total_score / len(ground_truth) * 100

    levenshtein_score = average_levenshtein_similarity(refs, preds)

    return {
        "accuracy (%)": round(accuracy, 2),
        "bleu": bleu_res,
        "rouge": rouge_res,
        "meteor": meteor_res,
        "jaccard (%)": round(jaccard, 2),
        "cosine (%)": round(cosine, 2),
        "levenshtein (%)": round(levenshtein_score, 2)
    }


In [27]:
metrics = evaluate_model(test_loader, model, processor)
for name, val in metrics.items():
    print(f"{name}: {val}")


Evaluating: 100%|██████████| 795/795 [05:10<00:00,  2.56it/s]
[nltk_data] Downloading package wordnet to /home/ebmi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ebmi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/ebmi/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


accuracy (%): 85.0
bleu: {'bleu': 0.5228646354456179, 'precisions': [0.9178678533423624, 0.8297338021094928, 0.6055776892430279, 0.4696969696969697], 'brevity_penalty': 0.7664125839723043, 'length_ratio': 0.789867807748147, 'translation_length': 10337, 'reference_length': 13087}
rouge: {'rouge1': np.float64(0.9177039980169297), 'rouge2': np.float64(0.16856280203567542), 'rougeL': np.float64(0.9157996556669944), 'rougeLsum': np.float64(0.9159090966573409)}
meteor: {'meteor': np.float64(0.5115662368102752)}
jaccard (%): 88.43
cosine (%): 77.92
levenshtein (%): 88.52
